Решить задачу машинного перевода выбрав свой язык:
* Формируем датасет с исходного языка на целевой (код прописать в классе)
* Строим архитектуру нейронной сети 
* Обучаем 
* Проверить качество с помощью метрики BLEU

In [1]:
from io import open
import unicodedata
import string
import re
import random
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import warnings

warnings.filterwarnings("ignore")
device = torch.device("cpu")

## Подготовка

In [2]:
# определим по умолчанию 3 токена которые будут нам информировать о начале предложения, конце предложения и неизвестном токене:
SOS_token = 0
EOS_token = 1
UNK_token = 2 # применяется разделение датасета на train и test части и в test могут быть токены, отсутствующие в train  

# Создадим объект словаря языка, который будет хранить данные по маппингу слов
class LanguageVocabulary(object):
    def __init__(self, name):
        # название языка
        self.name = name
        # словарик word2index который хранит соответственно кодировку слова в целочисленный индекс словаря
        self.word2index = {}
        # обычный словарик который хранит распределение слов, сколько слов мы использовали и сколько обнаружили
        self.word2count = {}
        # Обратный словарик словарю word2index где хранятся уже индексы и замаппенные слова к каждому индексу, нужен будет для расшифровки последовательности
        self.index2word = {0: "SOS", 1: "EOS", 2: "UNK"}
        # Count SOS, EOS и UNK, храним просто общее количество слов в нашем словаре, то есть количество токенов в сформированном словарике нашего языка
        self.n_words = 3

    def add_sentence(self, sentence):
        """
        Метод класса, для добавления предложения в словарь.
        Каждое предложение поступающее к нам, будет разбираться на
        примитивные токены и добавляться в словарь при помощи метода класса addword()
        """
        for word in sentence.split(' '):
            self.add_word(word)


    def add_word(self, word):
        # проверяем не входит ли наше слово в словарь word2index
        if word not in self.word2index:
            # добавляем в качестве ключа слово а в качестве значения последнее n_words
            self.word2index[word] = self.n_words
            # меняем на единичку
            self.word2count[word] = 1
            # и соответственно меняем и index2word словарик добавляя уже слово для декодирования
            self.index2word[self.n_words] = word
            # инкрементируем n_words
            self.n_words += 1
        else:
            # Если такое уже слово есть просто добавляем 1 что добавилось одно слово
            self.word2count[word] += 1

In [3]:
# http://stackoverflow.com/a/518232/2809427
def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalize_string(s):
    # декодируем из юникода в ascii
    s = unicode_to_ascii(s.lower().strip())
    # точку, !, ? меняем на пробел чтобы этот символ стоял отдельно от всех
    s = re.sub(r"([.!?])", r" \1", s)
    # оставляем только наборы символов указанных в паттерне регулярного выражения остальное заменим на пробел
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

Возьмем для перевода пару сильно отличающихся языков - английский язык/венгерский язык

In [4]:
def read_dict_sentences():
    # берем документ корпуса, он читается и разбивается на предложения
    print("Чтение строк...")
    lines = open('./input/task_3_dict.txt', encoding='utf-8').read().strip().split('\n')
    random.shuffle(lines) # перемешиваем строки
    # разбиваем построчно и нормализуем строку:
    pairs = [[normalize_string(s) for s in l.split('|')] for l in lines]
    
    # разделяем пары 80% на обучение, 20% на тест
    split_idx = int(len(pairs) * 0.8)
    train_pairs = pairs[:split_idx]
    test_pairs = pairs[split_idx:]
    
    print(f"Тренировочные данные {len(train_pairs)}, Тестовые данные: {len(test_pairs)}")
    return train_pairs, test_pairs

In [5]:
train_pairs, test_pairs = read_dict_sentences()

Чтение строк...
Тренировочные данные 10525, Тестовые данные: 2632


In [6]:
def read_eng_hun_vocabs(pairs, reverse=False):
    """
        Формируем словари английского и венгерского языков
    """
    lang1, lang2 = 'eng', 'hun'
    # Можем создавать и проходить как с целевого языка на исходный так и наоборот:
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = LanguageVocabulary(lang2)
        output_lang = LanguageVocabulary(lang1)
    else:
        input_lang = LanguageVocabulary(lang1)
        output_lang = LanguageVocabulary(lang2)
    return input_lang, output_lang, pairs

In [7]:
def prepare_data(reverse=False):
    input_lang, output_lang, pairs = read_eng_hun_vocabs(train_pairs, reverse)
    print("Прочитано %s пар фраз" % len(pairs))
    print("Подсчет слов...")
    for pair in pairs:
        input_lang.add_sentence(pair[0])
        output_lang.add_sentence(pair[1])
    print("Подсчитано слов:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [8]:
input_lang, output_lang, train_pairs = prepare_data()
print(random.choice(train_pairs))

Прочитано 10525 пар фраз
Подсчет слов...
Подсчитано слов:
eng 2176
hun 6205
['i am not lying', 'nem hazudok']


## Формирование Encoder

In [9]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        # hidden_size - размер скрытого состояния
        self.hidden_size = hidden_size
        # слой эмбеддингов, который из входного вектора последовательности отдаст представление последовательности для скрытого состояния
        # в качестве input_size - размер словаря
        self.embedding = nn.Embedding(input_size, hidden_size)
        # рекуррентная ячейка GRU которая принимает MxM (hidden на hidden)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        # приводим эмбеддинг к формату одного предлоежния 1х1 и любая размерность
        embedded = self.embedding(input).view(1, 1, -1)
        # нужно для следующего шага пока не запутываемся :) просто присвоили наш эмбеддинг
        output = embedded
        # и соответственно подаем все в ГРЮ ячейку (эмбеддинг и скрытые состояния)
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        # дополнительно сделаем инициализацию скрытого представления (просто заполним нулями)
        return torch.zeros(1, 1, self.hidden_size, device=device)

## Формирование Decoder

In [10]:
# максимальная длина последовательности
MAX_LENGTH = 50

In [11]:
class AttentionDecoder(nn.Module):
    
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttentionDecoder, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        attn_weights = F.softmax(
            self.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)
        attn_applied = torch.bmm(attn_weights.unsqueeze(0),
                                 encoder_outputs.unsqueeze(0))

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)

        output = F.relu(output)
        output, hidden = self.gru(output, hidden) 

        output = F.log_softmax(self.out(output[0]), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

## Вспомогательные функции для работы с парами фраз (предложений), и для кодирования, и подачи в модель

In [12]:
# токены кодируем в целочисленное представление; если токена нет, то возвращаем индекс метасимвола для отсутствующего токена
def indexesFromSentence(lang, sentence): 
    return [lang.word2index.get(word, UNK_token) for word in sentence.split(' ')]


# берем предложение с указанным языком, делаем из него индексы и вставляем метку конца предложения, превращаем в тензор:
def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

# для создания тензора из пар:
def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

## Функция обучения для работы только с одной парой

In [13]:
teacher_forcing_ratio = 0.5

def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    # инициализируем скрытое представление для энкодера
    encoder_hidden = encoder.initHidden()
    # скидываем градиенты для алгоритма градиентного спуска как и у энкодера так и у декодера
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    # получаем размер в словаря (токенов) для входящего и выходящего тензора так как мы пробегаемся по каждому предложению по кусочкам
    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    # создаем переменную где будем хранить наши выходы из энкодера (в данной реализации пока не юзаем, далее будет еще один вариант)
    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
    loss = 0
    # пробегаем по длине входящего тензора и в экодер передаем последовательно каждый из токенов:
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        # сохраняем все выходы из энкодера для одного слова (для передачи в декодер, ниже)
        encoder_outputs[ei] = encoder_output[0, 0]


    # закончили с энкодером пошли к декодеру, как было сказано декодер начинается с SOS
    decoder_input = torch.tensor([[SOS_token]], device=device)
    
    decoder_hidden = encoder_hidden

    # будем использовать Teacher Forcing в части случае (подставляя правильную последовательность)
    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False
    if use_teacher_forcing:
        # подаем decoder_input = torch.tensor([[SOS_token]], device=device) то есть по одному слову и скрытое представление
        for di in range(target_length):
            # переведенное предложение и скрытое представление
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            # считаем ошибку
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  
    else:
        for di in range(target_length):
            decoder_output, decoder_hidden, _ = decoder(decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach() 
            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break
    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
    return loss.item() / target_length

In [14]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / percent
    rs = es - s
    return '%s (- eta: %s)' % (asMinutes(s), asMinutes(rs))

## Функция обучения для всех тренировочных пар

In [15]:
def trainIters(encoder, decoder, n_iters, print_every=1000, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Обнуляется при каждом print_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    # делаем выборку тренировочных пар функцией которую создали до
    training_pairs = [tensorsFromPair(random.choice(train_pairs)) for i in range(n_iters)]
    # используем Negative Log-Likelihood Loss, т.к. log softmax уже присутствует в модели
    criterion = nn.NLLLoss()

    for epoch in range(1, n_iters + 1):
        training_pair = training_pairs[epoch - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]
        # используем функцию для тренировки на отдельных токенах, которую написали выше
        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_iters),
                                         epoch, epoch / n_iters * 100, print_loss_avg))

## Функция, позволяющая использовать Encoder-Decoder для перевода предложения

In [16]:
def evaluate_sentences(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            decoder_attentions[di] = decoder_attention.data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1]

In [17]:
import evaluate

bleu = evaluate.load("bleu")

def evaluateRandomly(pairs_to_evaluate, encoder, decoder, n=10):
    references, predictions = [],[]
    
    # для вывода 3 примеров перевода 
    for i in range(3):
        pair = random.choice(pairs_to_evaluate)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate_sentences(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

    # для вычисления bleu метрики 
    for i in range(n):
        pair = random.choice(pairs_to_evaluate)
        references.append(pair[1])
        output_words, _ = evaluate_sentences(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        predictions.append(output_sentence)

    results = bleu.compute(predictions=predictions, references=references)
    return results

## Этап обучения

In [18]:
hidden_size = 256
encoder1 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder1 = AttentionDecoder(hidden_size, output_lang.n_words, dropout_p=0.5).to(device)
trainIters(encoder1, decoder1, 90000, print_every=9000)

4m 19s (- eta: 38m 57s) (9000 10%) 4.5848
8m 36s (- eta: 34m 27s) (18000 20%) 3.3821
13m 13s (- eta: 30m 51s) (27000 30%) 2.5696
17m 43s (- eta: 26m 35s) (36000 40%) 2.0527
22m 26s (- eta: 22m 26s) (45000 50%) 1.6365
27m 49s (- eta: 18m 32s) (54000 60%) 1.3088
32m 46s (- eta: 14m 2s) (63000 70%) 1.0650
37m 22s (- eta: 9m 20s) (72000 80%) 0.8607
41m 57s (- eta: 4m 39s) (81000 90%) 0.7041
46m 54s (- eta: 0m 0s) (90000 100%) 0.5765


In [26]:
train_score = evaluateRandomly(train_pairs, encoder1, decoder1, n=500)
print(f"Значение BLEU метрики для тренировочных данных: {train_score['bleu']:.2f}") 

> this is stillness
= ez a csend
< ez a csend van <EOS>

> we are here with our people
= itt vagyunk a nepunkkel
< itt vagyunk a nepunkkel <EOS>

> they value their ancestors
= ertekelik az oseiket
< ertekelik az oseiket <EOS>

Значение BLEU метрики для тренировочных данных: 0.37


In [28]:
test_score = evaluateRandomly(test_pairs, encoder1, decoder1, n=500)
print(f"Значение BLEU метрики для тестовых данных: {test_score['bleu']:.2f}") 

> i am a bridge for those who come after
= hid vagyok az utanam jovoknek
< hid vagyok az utanad jovoknek <EOS>

> this is what love looks like
= igy nez ki a szeretet
< igy nez ki a kapcsolodas <EOS>

> she is rooted in her ancestors
= az oseiben gyokerezik
< az osei gyokerezik <EOS>

Значение BLEU метрики для тестовых данных: 0.24
